In [ ]:
import os, eda_analysis

cfg = eda_analysis.EdaConfig(family="lookahead/reward")
S   = eda_analysis.notebook_setup(cfg)

# `lookahead/reward` -- RQ-i: does K-turn look-ahead help, within each optimizer?

**The question.** Look-ahead changes only *what context the oracle scores*: with `K > 0` each
candidate completion is extended by K more simulated turns before the grader sees it, and nothing
about the loss changes. So the lever can be read inside PTO and inside GRPO separately, and the
comparison that isolates it is **K=5 versus K=0 holding the optimizer fixed**.

**THE SIGN CONVENTION, used in every table, figure and caption below.**

* `mean_delta = score(K_hi) - score(K_lo)` -- positive means the **K=5 arm scored HIGHER**.
  Higher, not better.
* `gain = sign_of(metric) * mean_delta` -- positive always means the K=5 arm was **BETTER**,
  including on MICI, which counts MI-INCONSISTENT behaviour and is therefore lower-is-better.
* CI ends are swapped when the sign flips, so `gain_ci_lo <= gain_ci_hi` on every row.

**The pairing unit is `persona_id`, always.** The same 96 personas face every arm and every
iteration, so this is a repeated-measures design: each contrast subtracts WITHIN persona and
analyses the deltas. Persona variance dominates the between-arm variance, so pairing is not a
refinement -- unpaired, a real effect can vanish into it. Pairing by row order would leave the
*mean* correct while making `dz`, the CI and the p-value meaningless, which is exactly why it is
never done here: `stats.paired_arrays` joins on `persona_id` and nothing else.

**Multiplicity.** Holm-Bonferroni is applied **within each grader**, over the family of rows in
that grader's endpoint table (every instrument x every optimizer). Two per-grader tables
concatenated are NOT jointly corrected; the family is stated on the table itself.

**Matched state.** The endpoint contrast is taken at the highest TRAINED model state both arms
have scored on every instrument, so the two sides are compared at the same iteration index. State
0 is reported separately as a **placebo**: both arms are the same untrained policy there, so any
non-zero contrast at state 0 is sampling noise and bounds what "no effect" looks like on this
instrument.

**Difference-in-differences.** The last section asks whether the K effect itself differs BY
METHOD: `(PTO_K5 - PTO_K0) - (GRPO_K5 - GRPO_K0)`, computed per persona so it is paired all the
way through. Positive means look-ahead bought PTO more than it bought GRPO.

In [ ]:
# ---------------------------------------------------------------------------
# Imports, both graders, the K-pairs, and the guards that let this notebook render
# with NO DATA on disk. Every section degrades to an explicit "no data yet" artifact.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from eda_analysis import constants, data, exports, plotting, stats

SEED = S.CFG.boot_seed
FOCUS = S.CFG.focus_metric

ALL = data.scores_by_judge(S.ARMS, rep=S.CFG.judge_rep, attach_persona=S.CFG.attach_persona)
if ALL.empty and not S.SCORES.empty:
    ALL = S.SCORES
HAVE = not ALL.empty

JUDGES = sorted(ALL["judge"].unique()) if HAVE else [S.JUDGE]
METRICS = [m for m in constants.METRIC_ORDER if HAVE and m in set(ALL["metric"].unique())]

NO_DATA = ("NO DATA YET -- no scored conversations for these arms. Generate the K=0 and K=5 arms "
           "of an optimizer, run notebooks/scoring/Run_Eval.ipynb, and re-render.")
NO_PAIR = ("NO K CONTRAST AVAILABLE -- this needs TWO arms of the same optimizer at different "
           "look-ahead depths (e.g. GRPO4_..._LA0_... and GRPO4_..._LA5_...). Only one depth is "
           "on disk.")

exports.reset_results()
exports.save_provenance(S.CFG, ALL)


def placeholder(name, message=NO_DATA, group=None, caption=None):
    fig = plt.figure(figsize=(7.6, 1.9))
    ax = fig.add_subplot(111)
    ax.axis("off")
    ax.text(0.5, 0.5, message, ha="center", va="center", fontsize=8.5,
            color="#777777", wrap=True)
    exports.save_fig(fig, name, group=group, caption=caption or message)
    plt.close(fig)


def k_pairs(arms):
    # One contrast per optimizer: its LARGEST look-ahead depth against its SMALLEST, so a
    # future K=3 arm pairs against K=0 instead of being silently dropped. Labels are display
    # keys, so two arms differing only in MCL would share one -- warned about below.
    by_method = {}
    for a in arms:
        by_method.setdefault(a.method, {}).setdefault(a.k, set()).add(a.label)
    out = []
    for method in sorted(by_method):
        ks = sorted(by_method[method])
        if len(ks) < 2:
            continue
        lo, hi = ks[0], ks[-1]
        for arm_lo in sorted(by_method[method][lo]):
            for arm_hi in sorted(by_method[method][hi]):
                out.append({"method": method, "k_lo": lo, "k_hi": hi,
                            "arm_lo": arm_lo, "arm_hi": arm_hi,
                            "label": f"{method}: K{hi} - K{lo}"})
    return out


PAIRS = k_pairs(S.ARMS)
labels = [a.label for a in S.ARMS]
if len(set(labels)) != len(labels):
    print("WARNING: two arms share a display label (they differ only in something the label "
          "elides -- MCL, branch width, rubric). Contrasts below would merge them; key on "
          "experiment_name if that is not what you want.")

print(f"judges : {JUDGES}")
print(f"metrics: {METRICS or '(none scored)'}")
print(f"K pairs: {[p['label'] for p in PAIRS] or '(none -- need two depths per optimizer)'}")
if not HAVE:
    print(NO_DATA)
elif not PAIRS:
    print(NO_PAIR)

## 1. The contrast machinery

`cell_scores` narrows the long frame to exactly one (grader, arm, model state, instrument) cell,
which is the shape `stats.paired_arrays` requires -- it *raises* on a frame that still holds
several instruments, because pairing that would form a cross product rather than pairs.

`contrast_row` is the only place a difference is computed in this notebook, so the sign convention
lives in one place: **`arm_a - arm_b`**, with `arm_a` always the higher-K arm.

In [ ]:
CONTRAST_COLS = ["judge", "contrast", "method", "metric", "instrument", "iteration",
                 "arm_a", "arm_b", "k_a", "k_b", "n", "mean_delta", "ci_lo", "ci_hi", "dz",
                 "gain", "gain_ci_lo", "gain_ci_hi", "gain_dz", "improved", "effect",
                 "t", "p", "p_holm", "stars", "sign"]


def cell_scores(df, judge, arm, state, metric):
    # ONE (grader, arm, model state, instrument) cell, as persona_id + score.
    m = df[(df["judge"] == judge) & (df["arm_label"] == arm)
           & (df["iteration"] == int(state)) & (df["metric"] == metric)]
    return m[["persona_id", "score"]].dropna(subset=["score"])


def states_for(df, judge, arm, metric):
    sub = df[(df["judge"] == judge) & (df["arm_label"] == arm) & (df["metric"] == metric)]
    return set(int(s) for s in sub["iteration"].unique())


def common_states(df, judge, metric, arms):
    common = None
    for arm in arms:
        s = states_for(df, judge, arm, metric)
        common = s if common is None else (common & s)
    return sorted(common or [])


def matched_state(df, judge, arms, metrics):
    # The highest TRAINED state every arm has scored on EVERY instrument, so all metrics of a
    # row are read at one iteration index. State 0 is excluded: it is the placebo, not an
    # endpoint.
    common = None
    for arm in arms:
        for metric in metrics:
            s = states_for(df, judge, arm, metric)
            common = s if common is None else (common & s)
    trained = sorted(x for x in (common or []) if x > 0)
    return trained[-1] if trained else None


def contrast_row(df, judge, metric, state, arm_a, arm_b, **extra):
    # SIGN: arm_a - arm_b. Positive mean_delta = arm_a scored HIGHER; `gain` re-signs it so
    # positive is BETTER on every instrument (MICI included). Paired on persona_id.
    a = cell_scores(df, judge, arm_a, state, metric)
    b = cell_scores(df, judge, arm_b, state, metric)
    if a.empty or b.empty:
        return None
    va, vb = stats.paired_arrays(a, b, on="persona_id", value="score")
    if va.size == 0:
        return None
    row = stats.orient_contrast(stats.paired_contrast(va, vb, seed=SEED), metric)
    row.update({"judge": judge, "metric": metric, "instrument": constants.short_label(metric),
                "iteration": int(state), "arm_a": arm_a, "arm_b": arm_b,
                "effect": stats.effect_label(row.get("gain_dz"))})
    row.update(extra)
    return row


def delta_rows(df, judge, metric, state, arm_a, arm_b):
    # The per-persona delta arm_a - arm_b, as a one-row-per-persona frame. Used both for the
    # by-state trajectory (whose CI is therefore a PAIRED interval) and for the DiD.
    a = cell_scores(df, judge, arm_a, state, metric)
    b = cell_scores(df, judge, arm_b, state, metric)
    if a.empty or b.empty:
        return pd.DataFrame({"persona_id": pd.Series(dtype="int64"),
                             "score": pd.Series(dtype="float64")})
    m = a.merge(b, on="persona_id", how="inner", suffixes=("_a", "_b"))
    return pd.DataFrame({"persona_id": m["persona_id"].to_numpy(),
                         "score": (m["score_a"] - m["score_b"]).to_numpy(dtype=float)})


def tidy_by_judge(rows):
    # Holm-Bonferroni WITHIN each grader: the family is that grader's rows and nothing else.
    if not rows:
        return pd.DataFrame()
    raw = pd.DataFrame([r for r in rows if r is not None])
    if raw.empty:
        return pd.DataFrame()
    parts = [stats.summarize_contrasts(g) for _, g in raw.groupby("judge", sort=True)]
    out = pd.concat(parts, ignore_index=True)
    lead = [c for c in CONTRAST_COLS if c in out.columns]
    return out[lead + [c for c in out.columns if c not in lead]]


print("contrast helpers ready")

## 2. The K contrast at the matched endpoint

One row per (grader, optimizer, instrument) at the matched final state. `n` is the number of
personas that survived the inner join -- if a state is only partly scored, this is the honest
sample size, not 96 by assumption.

Read `gain` with its CI, not `p`. A `gain_ci` that excludes zero is the claim; the stars are a
reading aid on the Holm-adjusted p, and the family that was adjusted is this grader's rows.

In [ ]:
rows = []
placebo = []
matched = {}
if HAVE and PAIRS and METRICS:
    for judge in JUDGES:
        for pair in PAIRS:
            arms = [pair["arm_hi"], pair["arm_lo"]]
            state = matched_state(ALL, judge, arms, METRICS)
            matched[(judge, pair["label"])] = state
            if state is None:
                continue
            for metric in METRICS:
                rows.append(contrast_row(
                    ALL, judge, metric, state, pair["arm_hi"], pair["arm_lo"],
                    contrast=pair["label"], method=pair["method"],
                    k_a=pair["k_hi"], k_b=pair["k_lo"]))
                if 0 in common_states(ALL, judge, metric, arms):
                    placebo.append(contrast_row(
                        ALL, judge, metric, 0, pair["arm_hi"], pair["arm_lo"],
                        contrast=pair["label"] + " @ base", method=pair["method"],
                        k_a=pair["k_hi"], k_b=pair["k_lo"]))

K_ENDPOINT = tidy_by_judge(rows)
K_PLACEBO = tidy_by_judge(placebo)

exports.save_table(
    K_ENDPOINT, "k_contrast_endpoint",
    caption=("RQ-i at the matched endpoint. One row per grader x optimizer x instrument, paired "
             "on persona_id at the highest trained model state both arms have scored. SIGN: "
             "mean_delta = K_hi - K_lo (higher, not better); gain = sign_of(metric) * mean_delta, "
             "so positive is BETTER on every instrument including MICI. CIs are 2,000-resample "
             "percentile bootstraps of the paired mean (seed=BOOT_SEED). p_holm is corrected "
             "WITHIN each grader over that grader's rows."))
exports.save_table(
    K_PLACEBO, "k_contrast_placebo_base",
    caption=("PLACEBO: the same K contrast at model state 0, where both arms are the SAME "
             "untrained policy and the only difference is which conversations were sampled. Any "
             "non-zero gain here is sampling noise, and it bounds what 'no effect' looks like on "
             "each instrument. Same sign convention and pairing unit as the endpoint table."))

if K_ENDPOINT.empty:
    print(NO_DATA if not (HAVE and PAIRS) else "no matched state shared by a K pair yet")
else:
    print({k: v for k, v in matched.items()})
    display(K_ENDPOINT[K_ENDPOINT["metric"] == FOCUS])

## 3. Forest plot of the endpoint contrasts

One dot and CI per (grader, optimizer, instrument). The x axis is the **raw** paired difference
`K_hi - K_lo`, so the plotted number matches the table; only the colour is oriented, which is why
a green dot on MICI sits at a *negative* x. Grey means the interval covers zero.

In [ ]:
if K_ENDPOINT.empty:
    placeholder("k_contrast_forest", caption="No K contrast to plot yet. " + NO_DATA)
else:
    forest = K_ENDPOINT.copy()
    forest["label"] = (forest["contrast"] + "  |  " + forest["instrument"]
                       + "  [" + forest["judge"] + "]")
    forest = forest.sort_values(["judge", "contrast", "metric"])
    fig = plotting.contrast_forest(
        forest, label_col="label", value_col="mean_delta", lo_col="ci_lo", hi_col="ci_hi",
        annot_col="dz", metric_col="metric",
        title="RQ-i: look-ahead K contrast at the matched endpoint",
        xlabel="paired difference K_hi - K_lo (95% bootstrap CI)")
    if fig is not None:
        exports.save_fig(
            fig, "k_contrast_forest",
            caption=("Paired K contrasts at the matched endpoint, one row per grader x optimizer "
                     "x instrument, pairing unit persona_id. The x axis is the RAW difference "
                     "K_hi - K_lo; colour is oriented by instrument, so on MICI (lower better) a "
                     "green dot lies at a negative x. Grey = the 95% bootstrap CI includes zero."))
        plt.close(fig)
    print(f"forest rendered with {len(forest)} rows")

## 4. The K effect across model states

The paired delta on the training-reward axis at **every** state both arms have scored, so the
lever can be read as a trajectory rather than at one endpoint. State 0 is the placebo described
above and should sit on the zero line.

Unlike the bands in `arms/outcomes`, this band **is** a paired interval: it bootstraps the
per-persona deltas, so the persona variance is already differenced out.

In [ ]:
records = []
if HAVE and PAIRS:
    for judge in JUDGES:
        for pair in PAIRS:
            for state in common_states(ALL, judge, FOCUS, [pair["arm_hi"], pair["arm_lo"]]):
                d = delta_rows(ALL, judge, FOCUS, state, pair["arm_hi"], pair["arm_lo"])
                if d.empty:
                    continue
                records.append(d.assign(judge=judge, iteration=int(state), metric=FOCUS,
                                        contrast=pair["label"]))

DELTA_LONG = (pd.concat(records, ignore_index=True) if records
              else pd.DataFrame(columns=["persona_id", "score", "judge", "iteration",
                                         "metric", "contrast"]))

by_state_rows = []
if not DELTA_LONG.empty:
    for (judge, contrast, state), g in DELTA_LONG.groupby(["judge", "contrast", "iteration"]):
        x = g["score"].to_numpy(dtype=float)
        lo, hi = stats.bootstrap_ci(x, np.mean, seed=SEED)
        sign = int(constants.sign_of(FOCUS))
        by_state_rows.append({"judge": judge, "contrast": contrast, "iteration": int(state),
                              "metric": FOCUS, "n": int(x.size),
                              "mean_delta": float(np.mean(x)), "ci_lo": lo, "ci_hi": hi,
                              "gain": sign * float(np.mean(x)),
                              "gain_ci_lo": lo if sign > 0 else -hi,
                              "gain_ci_hi": hi if sign > 0 else -lo,
                              "dz": stats.cohens_dz(x, np.zeros_like(x)), "sign": sign})

K_BY_STATE = (pd.DataFrame(by_state_rows).sort_values(["judge", "contrast", "iteration"])
              .reset_index(drop=True) if by_state_rows else pd.DataFrame())
exports.save_table(
    K_BY_STATE, "k_contrast_by_state",
    caption=(f"The paired K contrast on {FOCUS} at EVERY model state both arms have scored, per "
             f"grader. mean_delta = K_hi - K_lo; gain is signed so positive is better. CIs "
             f"bootstrap the per-persona DELTAS (seed=BOOT_SEED), so they are paired intervals. "
             f"State 0 is the untrained-base placebo. No multiplicity correction is applied "
             f"across states -- these are one trajectory, not a family of independent tests."))

for judge in JUDGES:
    name = f"k_effect_by_state_{judge}"
    sub = DELTA_LONG[DELTA_LONG["judge"] == judge] if not DELTA_LONG.empty else DELTA_LONG
    if sub.empty:
        placeholder(name, message=(NO_DATA if not PAIRS else NO_PAIR),
                    caption=f"No K contrast for grader {judge}.")
        continue
    fig = plotting.score_trajectory(
        sub, metric=FOCUS, metric_col="metric", x="iteration", y="score", arm_col="contrast",
        base_value=None, title=f"K effect on {constants.short_label(FOCUS)} ({judge})",
        xlabel="model state", ylabel=f"paired delta in {FOCUS}  (K_hi - K_lo)")
    plotting.add_base_line(fig.axes[0], 0.0, label="no effect")
    legend = fig.axes[0].get_legend()
    if legend is not None:
        legend.set_title("contrast")
    exports.save_fig(
        fig, name,
        caption=(f"Mean PAIRED delta ({FOCUS}, K_hi - K_lo) by model state, grader {judge}, "
                 f"pairing unit persona_id. The band bootstraps the per-persona deltas "
                 f"(seed=BOOT_SEED) and is therefore a paired interval, unlike the unpaired "
                 f"bands in arms/outcomes. The dotted line is no effect; state 0 is the "
                 f"untrained-base placebo. Positive = the higher-K arm scored higher "
                 f"({FOCUS} is higher-is-better)."))
    plt.close(fig)
print(f"{len(K_BY_STATE)} by-state rows")

## 5. Difference-in-differences -- does the K effect depend on the optimizer?

`(PTO_K5 - PTO_K0) - (GRPO_K5 - GRPO_K0)`, computed **per persona** so the whole quantity is
paired: each persona contributes one PTO delta and one GRPO delta, and the contrast is over those
96 differences of differences.

Positive means look-ahead bought the first optimizer more than it bought the second. This needs
all four arms scored at one shared state; with fewer, the table is empty and says so.

A DiD is a second-order quantity and its CI is correspondingly wider -- a null here is weak
evidence of equality, not evidence of it.

In [ ]:
did_rows = []
if HAVE and METRICS:
    pto = next((p for p in PAIRS if p["method"] == "PTO"), None)
    grpo = next((p for p in PAIRS if p["method"] == "GRPO"), None)
    if pto is not None and grpo is not None:
        quad = [pto["arm_hi"], pto["arm_lo"], grpo["arm_hi"], grpo["arm_lo"]]
        for judge in JUDGES:
            state = matched_state(ALL, judge, quad, METRICS)
            if state is None:
                continue
            for metric in METRICS:
                d_pto = delta_rows(ALL, judge, metric, state, pto["arm_hi"], pto["arm_lo"])
                d_grpo = delta_rows(ALL, judge, metric, state, grpo["arm_hi"], grpo["arm_lo"])
                if d_pto.empty or d_grpo.empty:
                    continue
                va, vb = stats.paired_arrays(d_pto, d_grpo, on="persona_id", value="score")
                if va.size == 0:
                    continue
                row = stats.orient_contrast(
                    stats.paired_contrast(va, vb, seed=SEED), metric)
                row.update({"judge": judge, "metric": metric,
                            "instrument": constants.short_label(metric),
                            "iteration": int(state),
                            "contrast": "(PTO K5-K0) - (GRPO K5-K0)",
                            "arm_a": f"{pto['arm_hi']} - {pto['arm_lo']}",
                            "arm_b": f"{grpo['arm_hi']} - {grpo['arm_lo']}",
                            "effect": stats.effect_label(row.get("gain_dz"))})
                did_rows.append(row)

DID = tidy_by_judge(did_rows)
exports.save_table(
    DID, "k_difference_in_differences",
    caption=("Difference-in-differences: (PTO K_hi - K_lo) - (GRPO K_hi - K_lo), per grader and "
             "instrument at the state all four arms share. Computed PER PERSONA -- each persona "
             "contributes one PTO delta and one GRPO delta -- so the contrast is paired "
             "throughout; n is the number of personas scored in all four cells. Positive gain = "
             "look-ahead bought PTO more than it bought GRPO. Holm within grader. A second-order "
             "quantity: a null here is weak evidence of equality, not evidence of it."))
if DID.empty:
    print("no difference-in-differences: needs all four arms (both optimizers x both depths) "
          "scored at one shared model state")
else:
    display(DID[DID["metric"] == FOCUS])

## 6. Number ledger and index

The citable form of RQ-i, per grader: the sign convention, the matched state, and the gain with
its CI on the training-reward axis. Nothing here is pooled across graders.

In [ ]:
values = {
    "rqi.sign_convention": {
        "value": "mean_delta = score(K_hi) - score(K_lo); gain = sign_of(metric) * mean_delta",
        "source": "tables/k_contrast_endpoint.md",
        "note": "positive gain = the higher-K arm was BETTER, on every instrument"},
    "rqi.pairing_unit": {"value": "persona_id", "source": "",
                         "note": "repeated measures; the same 96 personas face every arm"},
    "rqi.multiplicity": {"value": "Holm-Bonferroni within grader",
                         "source": "tables/k_contrast_endpoint.md",
                         "note": "family = that grader's rows (instruments x optimizers)"},
    "rqi.pairs": {"value": [p["label"] for p in PAIRS], "source": "", "note": ""},
    "rqi.matched_states": {"value": {f"{j} | {c}": v for (j, c), v in matched.items()},
                           "source": "tables/k_contrast_endpoint.md",
                           "note": "highest trained state both arms scored on every instrument"},
}
if not K_ENDPOINT.empty:
    focus_rows = K_ENDPOINT[K_ENDPOINT["metric"] == FOCUS]
    for _, r in focus_rows.iterrows():
        key = f"rqi.{r['judge']}.{r['contrast'].replace(' ', '')}.{FOCUS}"
        values[key] = {
            "value": float(r["gain"]),
            "source": "tables/k_contrast_endpoint.md",
            "note": (f"gain (positive = higher K better) at state {int(r['iteration'])}, "
                     f"n={int(r['n'])} personas, 95% CI "
                     f"[{float(r['gain_ci_lo']):.3f}, {float(r['gain_ci_hi']):.3f}], "
                     f"dz={float(r['gain_dz']):.3f}, p_holm={float(r['p_holm']):.4f}")}
if not DID.empty:
    for _, r in DID[DID["metric"] == FOCUS].iterrows():
        values[f"rqi.did.{r['judge']}.{FOCUS}"] = {
            "value": float(r["gain"]),
            "source": "tables/k_difference_in_differences.md",
            "note": (f"(PTO K effect) - (GRPO K effect) at state {int(r['iteration'])}, "
                     f"n={int(r['n'])}, 95% CI [{float(r['gain_ci_lo']):.3f}, "
                     f"{float(r['gain_ci_hi']):.3f}]")}

exports.save_numbers(
    "lookahead_reward", values,
    caption=("Citable RQ-i numbers, per grader, with the sign convention and pairing unit stated "
             "alongside. Never pooled across graders."))
print(exports.build_index())